# Phase 3+4 integration — n=10 verification with Saighi A_k self-inhibition (Colab, parallel)

Runs `experiments/19_phase34_integrated.py` for **10 seeds** {17, 11, 23, 1, 2, 3, 5, 7, 13, 19} on CUDA with `--inhibition-gain 0.01 --n-cues 3000`.

**Parallel execution.** Per-seed runtime is ~17 min sequential because the workload is CPU-bound (per-cue Python loops in the Hebbian updater + checkpoint evaluation), not GPU-bound. GPU memory usage per seed is ~600 MB, well under A100's 40 GB. We launch all 10 seeds simultaneously as independent Python processes; they share the GPU and saturate the CPU. Expected wall-clock: ~25-40 min total instead of ~170 min sequential.

**Why n=10 and n_cues=3000:** Report 032 already showed n=5 has been *sample-lucky* twice (029, 034-seed-1). Report 033 telemetry: patterns only reach equilibrium at cue ~1200, so n_cues=1500 cuts off right after equilibrium; n_cues=3000 gives 1500 cues of equilibrium-state dynamics.

**Verifies report 034's seed-1 result on a serious n.** Single-seed showed top1 +0.009, R@10 +0.022, cap_t05 +0.022.

**What we're watching:** 8/10+ seeds with ΔR@10 ≥ 0; seed 23 (persistent negative outlier); Δtop1 distribution (blocker #6'); Δcap_t05 (blocker #3); A_k accumulation pattern in `death_diag`.

Output paths use `saighi_n3k` tag.

In [ ]:
# 1. Clone the repo. The Saighi A_k code must be pushed to main first.
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git log --oneline -5

In [ ]:
# 2. Mount Drive and copy the codebook into place.
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = '/content/drive/MyDrive/neuro-ai/phase3c_codebook_reconstruction.pt'
dst_dir = 'reports/phase3c_reconstruction'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(src, f'{dst_dir}/phase3c_codebook_reconstruction.pt')
!ls -lh {dst_dir}/phase3c_codebook_reconstruction.pt

In [ ]:
# 3. Install deps. Colab ships torch+CUDA; we just need datasets (for wikitext).
!pip install -q datasets

In [ ]:
# 4. Sanity check: GPU present, FHRR substrate + codebook load, A_k mechanism wired.
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
import os
print('cpu count (os):', os.cpu_count())

import sys; sys.path.insert(0, 'src')
from energy_memory.phase2.persistence import load_codebook
cb = load_codebook('reports/phase3c_reconstruction/phase3c_codebook_reconstruction.pt', device='cuda')
print('codebook:', cb.shape, cb.dtype, cb.device)

# Verify A_k mechanism is present in this checkout (otherwise the run will
# behave like baseline despite --inhibition-gain).
from energy_memory.phase4.consolidation import ConsolidationConfig, ConsolidationState
cfg = ConsolidationConfig(m=4, alpha=0.25, inhibition_gain=0.01)
s = ConsolidationState(cfg, device='cuda')
s.add_pattern()
s.accumulate_inhibition(0)
assert abs(s.A[0].item() - 0.01) < 1e-6, 'A_k not wired — code is stale, abort!'
print('A_k mechanism present, accumulate_inhibition works:', s.A[0].item())

In [ ]:
# 4b. Pre-warm the HuggingFace wikitext cache by loading it once in this process.
# If we don't do this, all 10 parallel subprocesses race to download/cache the
# same dataset simultaneously, which is slow and can corrupt the cache.
print('warming wikitext cache...')
from energy_memory.phase2.corpus import load_corpus_splits
from pathlib import Path
splits = load_corpus_splits('wikitext', Path('.'), wikitext_name='wikitext-2-raw-v1')
print('  train:', len(splits['train']))
print('  validation:', len(splits['validation']))
print('cache warmed.')

In [ ]:
# 5. PARALLEL launch: 10 seeds as independent subprocesses, all sharing the
# single A100 GPU. Each process uses ~600MB of VRAM and ~1 CPU thread.
# We launch all 10 at once and poll for completion.
import subprocess, os, time
from pathlib import Path

os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
RUN_TAG = 'saighi_n3k'
SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 19]
log_root = Path(f'reports/phase34_{RUN_TAG}_colab')
log_root.mkdir(parents=True, exist_ok=True)

def launch(seed):
    out_dir = f'reports/phase34_{RUN_TAG}_seed{seed}'
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    log_path = log_root / f'seed{seed}.log'
    logf = open(log_path, 'w')
    proc = subprocess.Popen(
        ['python', 'experiments/19_phase34_integrated.py',
         '--updater-kind', 'hebbian',
         '--seed', str(seed),
         '--n-cues', '3000',
         '--checkpoint-every', '500',
         '--success-threshold', '0.3',
         '--death-threshold', '0.05',
         '--death-window', '10',
         '--inhibition-gain', '0.01',
         '--inhibition-decay', '0.0',
         '--no-reencode-discovered',
         '--output-dir', out_dir],
        stdout=logf, stderr=subprocess.STDOUT,
    )
    return proc, logf, out_dir

t0 = time.time()
procs = {seed: launch(seed) for seed in SEEDS}
print(f'launched {len(procs)} parallel seeds at {time.strftime("%H:%M:%S")}')
for seed in SEEDS:
    print(f'  seed {seed}: pid={procs[seed][0].pid}, log={log_root}/seed{seed}.log')
print()

# Poll until all finish. Print each as it completes.
remaining = dict(procs)
while remaining:
    done_this_round = []
    for seed, (proc, logf, out_dir) in remaining.items():
        rc = proc.poll()
        if rc is not None:
            logf.close()
            elapsed = (time.time() - t0) / 60
            ok = '✓' if rc == 0 else f'FAILED (exit={rc})'
            json_exists = Path(out_dir, 'phase34_results.json').exists()
            print(f'[{elapsed:5.1f} min] seed {seed:>2}: {ok}  json={json_exists}')
            done_this_round.append(seed)
    for seed in done_this_round:
        del remaining[seed]
    if remaining:
        time.sleep(30)

print(f'\nALL DONE in {(time.time()-t0)/60:.1f} min')
# Show GPU utilization snapshot for sanity (and to know if we could push further)
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv

In [ ]:
# 6. Copy results back to Drive. Claude will pull from there for analysis.
import shutil, os
RUN_TAG = 'saighi_n3k'
SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 19]
dst = '/content/drive/MyDrive/neuro-ai/results'
os.makedirs(dst, exist_ok=True)
for seed in SEEDS:
    src = f'reports/phase34_{RUN_TAG}_seed{seed}'
    if os.path.isdir(src):
        shutil.copytree(src, f'{dst}/phase34_{RUN_TAG}_seed{seed}', dirs_exist_ok=True)
shutil.copytree(f'reports/phase34_{RUN_TAG}_colab',
                f'{dst}/phase34_{RUN_TAG}_colab', dirs_exist_ok=True)
print('results in', dst)
!ls {dst} | grep -E 'saighi_n3k'

In [ ]:
# 7. In-notebook summary table (per-seed + aggregate). Claude will re-do this with the raw JSONs but this lets you spot anomalies live.
import json
from pathlib import Path
import statistics as st

RUN_TAG = 'saighi_n3k'
SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 19]
rows = []
print(f'{"seed":>4} {"cond":>16} {"top1":>8} {"R@10":>8} {"cap_t05":>8} {"cands":>6} {"cons":>5} {"deaths":>6} {"A_max":>8} {"A_nz":>5}')
for seed in SEEDS:
    p = Path(f'reports/phase34_{RUN_TAG}_seed{seed}/phase34_results.json')
    if not p.exists():
        print(f'{seed:>4} MISSING')
        continue
    d = json.loads(p.read_text())
    A = B = C = None
    for cond in ('baseline_static', 'phase3_reencode', 'phase3_phase4'):
        f = d['results'][cond][-1]
        cands = f.get('candidates_total', 0)
        cons_ = f.get('consolidations', 0)
        deaths = f.get('deaths_total', 0)
        dd = f.get('death_diag', {}).get('2', {})
        A_max = dd.get('inhibition_max', 0.0)
        A_nz  = dd.get('inhibition_nonzero', 0)
        print(f'{seed:>4} {cond:>16} {f["top1"]:>8.4f} {f["topk"]:>8.4f} {f["cap_t_05"]:>8.4f} '
              f'{cands:>6} {cons_:>5} {deaths:>6} {A_max:>8.3f} {A_nz:>5}')
        if cond == 'baseline_static':  A = f
        elif cond == 'phase3_reencode': B = f
        elif cond == 'phase3_phase4':   C = f
    if A and B and C:
        rows.append({'seed': seed,
                     'dR10_CA': C['topk']-A['topk'],
                     'dR10_CB': C['topk']-B['topk'],
                     'dtop1_CA': C['top1']-A['top1'],
                     'dcap_CA': C['cap_t_05']-A['cap_t_05']})

print()
print(f'=== aggregate across {len(rows)} seeds ===')
tcrit = {9: 2.262, 8: 2.306, 7: 2.365, 6: 2.447, 5: 2.571, 4: 2.776}
df = max(1, len(rows)-1)
tc = tcrit.get(df, 2.262)
for k in ['dR10_CA','dR10_CB','dtop1_CA','dcap_CA']:
    vals = [r[k] for r in rows]
    m = st.mean(vals); sd = st.stdev(vals) if len(vals)>1 else 0.0
    se = sd / (len(vals)**0.5) if len(vals)>1 else 0.0
    lo, hi = m - tc*se, m + tc*se
    pos = sum(1 for v in vals if v > 0)
    print(f'  {k}: mean={m:+.4f}, 95%CI [{lo:+.4f}, {hi:+.4f}], pos/total={pos}/{len(rows)}, '
          f'per-seed={[f"{v:+.4f}" for v in vals]}')